In [68]:
import numpy as np
import pandas as pd 
import json
import re
import string
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from nltk.corpus import stopwords


In [69]:
df=pd.read_csv('D:\CINEIQ\CINEIQ\Datasets\imdb-dataset-of-50k-movie-reviews\IMDB Dataset.csv')

In [70]:
df['review'] = df['review'].str.replace(r'<br\s*/?>', '', regex=True)

In [71]:
df['review'] = df['review'].str.lower()

In [72]:
class SentimentThresholdOptimizer:

    def __init__(self, sia, save_path="best_threshold.json"):
        self.save_path = save_path
        self.threshold = None
        self.sia = sia
        self.load()

    def clean_text(self, text):
        # Remove punctuation
        text = text.translate(str.maketrans('', '', string.punctuation))

        # Remove stopwords
        stop_words = set(stopwords.words('english'))  # faster lookup
        words = [word for word in text.split() if word not in stop_words]

        return " ".join(words)
    
    # Train and update threshold (manual)
    def fit(self, df):
        cutpoint = []

        df1 = df.dropna().copy()
        df1['review'] = df1['review'].apply(self.clean_text)
        df1['compound'] = df1['review'].apply(lambda x: self.sia.polarity_scores(x)['compound'])

        for i in np.linspace(-1, 1, 40):
            df1['vader_sentiment'] = df1['compound'].apply(
                lambda x: "positive" if x > i else "negative"
            )
            accuracy = (df1['sentiment'] == df1['vader_sentiment']).mean()
            cutpoint.append((accuracy, i))

        best = max(cutpoint)
        self.threshold = best[1]

        self._save()
        print(f"Updated threshold: {self.threshold}")

        return self.threshold

    # Predict from compound score
    def predict_score(self, score):
        if self.threshold is None:
            raise ValueError("Threshold not available. Run fit() once.")
        return "positive" if score > self.threshold else "negative"
    
    def get_score(self, text):
        if not isinstance(text, str) or text.strip() == "":
            return None

        return self.sia.polarity_scores(self.clean_text(text))['compound']
    
    # NEW: Predict directly from text
    def predict_sentiment(self, text):
        if self.threshold is None:
            raise ValueError("Threshold not available. Run fit() once.")

        if not isinstance(text, str) or text.strip() == "":
            return {"Sentiment": None, "Score": None, "Text": text}

        score = self.sia.polarity_scores(self.clean_text(text))['compound']
        sentiment = "positive" if score > self.threshold else "negative"

        return {
            "Sentiment": sentiment,
            "Score": score,
            "Text": text
        }

    # NEW: Batch prediction
    def predict_batch(self, texts):
        results = []
        for text in texts:
            results.append(self.predict_sentiment(text))
        return pd.DataFrame(results)

    # Save threshold
    def _save(self):
        with open(self.save_path, "w") as f:
            json.dump({"threshold": self.threshold}, f)

    # Load threshold
    def load(self):
        try:
            with open(self.save_path, "r") as f:
                self.threshold = json.load(f)["threshold"]
        except:
            self.threshold = None                                    

In [73]:
sia = SentimentIntensityAnalyzer()
predictor=SentimentThresholdOptimizer(sia)

In [74]:
predictor.fit(df)

Updated threshold: 0.7948717948717947


np.float64(0.7948717948717947)

In [75]:
predictor.predict_batch(df['review'].head(10))

,Sentiment,Score,Text
0,negative,-0.9945,one of the other reviewers has mentioned that ...
1,positive,0.9545,a wonderful little production. the filming tec...
2,positive,0.9520,i thought this was a wonderful way to spend ti...
3,negative,-0.9117,basically there's a family where a little boy ...
4,positive,0.9871,"petter mattei's ""love in the time of money"" is..."
5,positive,0.9584,"probably my all-time favorite movie, a story o..."
6,positive,0.9246,i sure would like to see a resurrection of a u...
7,positive,0.9662,"this show was an amazing, fresh & innovative i..."
8,negative,-0.7074,encouraged by the positive comments about this...
9,positive,0.8979,if you like original gut wrenching laughter yo...


In [77]:
df.head(10)

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production. the filming tec...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically there's a family where a little boy ...,negative
4,"petter mattei's ""love in the time of money"" is...",positive
5,"probably my all-time favorite movie, a story o...",positive
6,i sure would like to see a resurrection of a u...,positive
7,"this show was an amazing, fresh & innovative i...",negative
8,encouraged by the positive comments about this...,negative
9,if you like original gut wrenching laughter yo...,positive
